# Healthcare Provider Fraud Detection — Exploratory Data Analysis (EDA)

This notebook analyzes the Medicare claims dataset aggregated to the provider level.
The objective is to identify key behavioral, clinical, and financial disparities between fraudulent and legitimate healthcare providers.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_parquet('../models/features.parquet')
print('Providers:', df.shape[0], 'Features:', df.shape[1] - 1)
df.head()

## 1. Class Distribution Analysis
Checking the degree of class imbalance in Medicare provider fraud labels.

In [2]:
counts = df['y'].value_counts()
pcts = counts / len(df) * 100
print(f'Class balance:\nNon-fraud: {counts[0]} ({pcts[0]:.2f}%)\nFraud: {counts[1]} ({pcts[1]:.2f}%)')

plt.figure(figsize=(6, 4))
bars = plt.bar(['Non-Fraud (0)', 'Fraud (1)'], counts, color=['#2ca02c', '#d62728'])
plt.title('Provider Class Balance')
plt.ylabel('Count')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

## 2. Feature Distribution Disparities
Comparing fraudulent vs. non-fraudulent distributions on high-impact predictors:
- Average reimbursement (`avg_reimb`)
- Total claim count (`claim_count`)
- Average claim duration (`avg_claim_duration`)
- Inpatient ratio (`inpatient_ratio`)

In [3]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
cols = ['avg_reimb', 'claim_count', 'avg_claim_duration', 'inpatient_ratio']
for ax, col in zip(axes.flatten(), cols):
    sns.kdeplot(data=df, x=col, hue='y', common_norm=False, ax=ax, palette=['#2ca02c', '#d62728'])
    ax.set_title(f'Distribution: {col}')
    ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 3. Correlation Analysis
Examining the correlation matrix among top features most correlated with provider fraud.

In [4]:
top15 = df.corr()['y'].abs().sort_values(ascending=False).head(16).index
plt.figure(figsize=(10, 8))
sns.heatmap(df[top15].corr(), annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Top 15 Feature Correlation Matrix')
plt.show()

## 4. Key Insights Summary

- **Severe Class Imbalance (~9% Fraud):** Only ~9% of providers are flagged as potentially fraudulent. Standard accuracy is an uninformative metric; PR-AUC and Recall-at-Precision are critical.
- **High Billing Volume Disparity:** Fraudulent providers process significantly higher claim counts and total reimbursement sums compared to normal providers, indicating systematic over-submission.
- **Per-Beneficiary Disproportion:** Fraudulent providers exhibit substantially higher `reimb_per_bene` and `claims_per_bene`, pointing to potential phantom billing or patient churning.
- **Inpatient and Duration Outliers:** Inpatient ratio and claim duration variance correlate positively with fraud, reflecting complex, high-payout claim manipulations.
- **Physician Network Signals:** Elevated claims-per-physician ratios highlight potential collusion rings or automated billing templates across concentrated attending physicians.